# SI4006 · M5 · Amparo — **Destilar un componente** (S14)

**Tópicos Especiales y Aplicaciones en IA** · Universidad EAFIT · Módulo 5

---

Sigue el lab de la profe (`sesiones/s14/S14_Lab_Destilacion_RESUELTO.ipynb`), **Parte B · datos sintéticos + LoRA**:
el profesor escribe el dataset, el estudiante se fine-tunea con la receta de S04 y se mide con **nuestro harness de M2**.

- **Profesor:** el brazo `4 · agente + reglas` de M3 (gpt-oss-120b por Groq, 4.2 llamadas por caso de media en M3).
- **Estudiante:** `Qwen2.5-0.5B-Instruct` + LoRA, una sola llamada, sin herramientas, en la misma GPU que el VLM.
- **Qué aprende:** solo el **JSON de hechos**. El mecanismo lo sigue decidiendo el motor del art. 6 (`reglas.decidir`):
  el estudiante no tiene que saber derecho y no puede inventar normas.

La lógica vive en `amparo/pipelines/m5_destilacion.py`; este notebook solo la llama y muestra lo que sale.

> GPU **T4** y `GROQ_API_KEY` en los secretos de Colab. Lo caro es Groq (~5 k tokens por relato que etiqueta el profesor):
> todo se guarda caso a caso y **se retoma donde quedó** si se acaba el cupo diario.

## 0 · Setup

In [ ]:
%pip install -q "transformers>=4.45" datasets accelerate peft scikit-learn pandas groq sentence-transformers
# Colab trae torchao 0.10 y peft reciente exige >= 0.16 si lo encuentra instalado ("incompatible
# version of torchao"). Aqui no se usa (es para cuantizar: S15), asi que se quita.
# Si ya habias importado peft en esta sesion: Entorno de ejecucion -> Reiniciar sesion.
%pip uninstall -q -y torchao
print('Listo.')

Listo.


In [ ]:
# Colab arranca vacío: se clona el repo (idempotente). Si el notebook ya corre dentro del repo, usa esa copia.
import os, sys, subprocess
from pathlib import Path

REPO_URL, RAMA = 'https://github.com/TomasPosada0626/Amparo.git', 'm5'
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'amparo' / 'nucleo' / 'schema.py').exists()), None)
if RAIZ is None:
    if not Path('Amparo').is_dir():
        subprocess.run(['git', 'clone', '-q', '-b', RAMA, REPO_URL, 'Amparo'], check=True)
    RAIZ = Path('Amparo').resolve()
sys.path.insert(0, str(RAIZ))

# Los intermedios caros (relatos, pares, caché del profesor, adaptador) van a Drive para sobrevivir al reinicio.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CARPETA = '/content/drive/MyDrive/amparo_m5'
except ImportError:
    CARPETA = None                      # local: runs/m5/ (ignorada por git)

from amparo.pipelines import m5_destilacion as m5

Mounted at /content/drive


## 1 · ¿Qué componente? El scorecard de M4

Slide 15, paso 1: *"miren su scorecard de M4: la columna `segundos` dice dónde se va el tiempo"*.
`segundos` es VLM + M3 por caso; las trazas guardan aparte lo que tardó el VLM (`segundos_ver`). La resta es M3.

In [ ]:
try:
    comp = m5.latencia_por_componente()
except FileNotFoundError as ex:      # la corrida de demo de M4 vive en main
    print(ex)

corrida de M4: m4/corridas/20261101T203828Z_demo
componente                                     mediana    p95    min    max  parte
VLM Qwen2-VL-2B (lectura del documento)           3.72   4.44   2.71   4.44    34%
M3 brazo 4 (agente gpt-oss-120b + reglas)         6.39  14.84   3.28  14.84    59%
total por caso                                   10.83  17.83   7.00  17.83   100%

mas lento: M3 brazo 4 (agente gpt-oss-120b + reglas)
mas variable: M3 brazo 4 (agente gpt-oss-120b + reglas)


**Lectura.** En la corrida de demo (L4, 11 casos) M3 se lleva ~59 % de la mediana (6.4 s de 10.8 s) y **toda la variación**
(de 3.3 a 14.8 s, según cuántos pasos dé el agente). El VLM tarda ~3.7 s y casi no varía.

Slide 15, paso 2: *¿esa pieza necesita ser grande?* El agente usa un modelo de 117 B para **marcar 13 booleanos y tres frases**:
la decisión jurídica ya la toma el motor de reglas. No necesita ser grande.
Paso 3: es el generador → **forma B**. La A (logits) no aplica: el profesor es una API y no da logits
(slide 12: la B *"sirve con una API cerrada"*).

### El contrato (TODO 1), antes de entrenar

Se copia al README de M5. *"Si no lo escriben antes, después cualquier resultado parece bueno."*

**Temperatura.** En la parte B no hay temperatura de destilación (esa es la T de la KL de la parte A, que necesita logits).
Hay dos temperaturas de **generación**, elegidas para bajar el riesgo:
- el profesor **escribe relatos** a `T = 0.7`, como en el lab → variedad;
- el profesor **responde** a `T = 0` con `seed = 42` (ya es así en `agente.py`) → la etiqueta es reproducible y no se le enseña ruido al estudiante;
- el estudiante responde **codicioso** (`do_sample=False`), como el `preguntar` de S04.

In [ ]:
ctx = m5.preparar(CARPETA)          # raíz, carpeta de trabajo, clave de Groq y el contrato

raiz: /content/Amparo
carpeta de trabajo: /content/drive/MyDrive/amparo_m5
clave de Groq cargada desde los secretos de Colab
device: cuda · NVIDIA A100-SXM4-40GB
CONTRATO: {
 "componente": "extractor de hechos de M3 (brazo 4: agente gpt-oss-120b por Groq)",
 "metrica": "acierto juridico del harness de M2 sobre los 130 casos de M3 (eval_set_v3 + eval_set_m3)",
 "acepto_perder_puntos": 5,
 "alucinacion_maxima": 0.0,
 "acepto_perder_puntos_adversarial": 8,
 "quiero_ganar": "M3 al menos 3x mas rapido (mediana por caso) y el 95 % de los casos en 5 s o menos, sin depender de la API",
 "quiero_ganar_x": 3.0,
 "p95_maximo_s": 5.0
}
corrida: demo-final-7-gf8c0e6c · NVIDIA A100-SXM4-40GB


## 2 · B1 · El profesor escribe los pares

Primero arma el profesor: el brazo 4 real de M3, igual que en M4. Después escribe relatos nuevos
(la celda B1 del lab con `TEMAS` = las 131 categorías del dataset de M1) y los responde.
Los relatos que se parecen demasiado al eval set se descartan (fuga).

In [ ]:
m5.armar_profesor(ctx);

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

profesor listo: 4 · agente + reglas · openai/gpt-oss-120b


In [ ]:
rel = m5.escribir_relatos(ctx)            # N_POR_TEMA = 2 → ~262 relatos; se retoma si se corta
for r in rel[:3]:
    print(f"[{r['categoria']}] {r['relato']}\n")

131 temas · 2 por tema · ya hechos 0 · faltan 262

262 relatos sinteticos · 0 descartados por parecerse al eval set (coseno >= 0.5) · coseno maximo que queda: 0.18
[ab_consulta_comercial] Tengo una casa en el barrio La Esperanza, en el tercer piso, con tres alcobas y un pequeño balcón. Hace poco mi primo me pidió que le ayude a fijar el precio del arriendo, pero yo no tengo idea de cuánto cobrar. Además, el edificio tiene una zona de juegos para niños que también quiero incluir en el cálculo. ¿Qué valor debería pedir?

[ab_consulta_comercial] Tengo una casa en la zona de la 68, en el tercer piso, con tres cuartos y un balcón. Quiero venderla pero no sé cuánto ponerle de precio. Además, el barrio ha tenido varios proyectos de construcción últimamente y no sé si eso afecta el valor. ¿Qué me recomendarían cobrar por ella?

[ab_hechos_insuficientes] Fui a la alcaldía del municipio para solicitar una copia de mi certificado de nacimiento y la funcionaria me respondió con un tono sarcástico,

In [ ]:
pares = m5.etiquetar_con_profesor(ctx)    # sintéticos + los 160 de train/validation de M1
import json
for p in pares[:3]:
    print(p['relato'][:160], '\n  →', json.dumps(p['objetivo'], ensure_ascii=False), '\n')

422 relatos · etiquetados 336 · faltan 86

422 pares (385 train · 37 validation) · 0 descartados
Tengo una casa en el barrio La Esperanza, en el tercer piso, con tres alcobas y un pequeño balcón. Hace poco mi primo me pidió que le ayude a fijar el precio de 
  → {"medio_eficaz": false, "derecho": "ninguno", "accionado": "autoridad o particular accionado", "pretension": "amparar el derecho invocado", "caso_triable": false, "motivo_no_triable": "(a) no es un caso legal, es una consulta de carácter informativo sin pretensión judicial"} 

Tengo una casa en la zona de la 68, en el tercer piso, con tres cuartos y un balcón. Quiero venderla pero no sé cuánto ponerle de precio. Además, el barrio ha t 
  → {"medio_eficaz": false, "derecho": "ninguno", "accionado": "autoridad o particular accionado", "pretension": "amparar el derecho invocado", "caso_triable": false, "motivo_no_triable": "a) la consulta es de carácter comercial y no plantea una controversia jurídica"} 

Fui a la alcaldía del mun

### Etiqueta dura vs. etiqueta del profesor

La celda de la profe compara la etiqueta del dataset con la del profesor. Aquí: con los hechos del **profesor**,
¿cuántas veces el motor llega al mecanismo gold en los relatos de M1? Lo que el profesor falla es ruido que el estudiante
va a heredar (slide 19).

In [ ]:
acuerdo = m5.acuerdo_profesor_gold(ctx)

el profesor (con el motor) acierta el mecanismo gold en 100/142 relatos de M1 (70%). Por mecanismo: {'accion_cumplimiento': '10/17', 'accion_popular': '11/16', 'habeas_corpus': '16/17', 'tutela': '43/62', 'via_ordinaria': '20/30'}


## 3 · B2 · El estudiante: Qwen2.5-0.5B + LoRA (receta de S04)

`r=8`, `lora_alpha=16`, `q_proj`/`v_proj`, `lr=2e-4`, 3 épocas. De M1 se conserva lo que ya se pagó: pérdida solo en la
respuesta, evaluación por época contra validation y el mejor checkpoint. Si el adaptador ya está en la carpeta de trabajo,
se carga en vez de reentrenar.

In [ ]:
if (ctx.carpeta / 'estudiante_lora').exists() and (ctx.carpeta / 'entrenamiento.json').exists():
    m5.cargar_estudiante(ctx)
else:
    m5.entrenar_estudiante(ctx)
print(json.dumps(ctx.entrenamiento.get('seleccion', {}), ensure_ascii=False, indent=1))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

... (2 outputs left out)

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093
train 385 · validation 37 · max_len 727
Traducidos a un parametro que esta version SI acepta:
   warmup_ratio->warmup_steps: {'ratio': 0.05, 'pasos': 7, 'pasos_totales_estimados': 144}
early stopping activo: para tras 2 epocas sin mejorar `eval_loss` (techo 3)


Epoch,Training Loss,Validation Loss
1,0.510737,0.648100
2,0.445656,0.584994
3,0.398151,0.577170



  eval_loss por epoca: 1:0.6481  2:0.5850  3:0.5772
  minimo en la epoca 3 (0.5772)  ultima 3 (0.5772)
  El minimo esta en la ultima epoca: el presupuesto no sobraba,
  y puede que falte. Vale la pena probar con mas.
estudiante: 494 M parametros · entrenado en 2.4 min
{
 "epocas_configuradas": 3,
 "curva_eval_loss": [
  {
   "epoca": 1.0,
   "eval_loss": 0.6481
  },
  {
   "epoca": 2.0,
   "eval_loss": 0.58499
  },
  {
   "epoca": 3.0,
   "eval_loss": 0.57717
  }
 ],
 "mejor_epoca": 3.0,
 "mejor_eval_loss": 0.57717,
 "ultima_epoca": 3.0,
 "ultima_eval_loss": 0.57717,
 "las_epocas_sobraban": false,
 "cargado": "el mejor checkpoint (load_best_model_at_end=True)"
}


In [ ]:
# El estudiante sobre 3 relatos de validation, junto a lo que dijo el profesor
for p in [p for p in ctx.pares if p['split'] != 'train'][:3]:
    print(p['relato'][:140])
    print('  profesor  :', m5.a_texto(p['objetivo']))
    print('  estudiante:', ctx.generar(p['relato']), '\n')

Me autorizaron una cirugía de vesícula hace seis meses y cada vez que llamo a la EPS me dicen que no hay agenda disponible. Ya fui dos veces
  profesor  : {"titular_pide_lo_suyo":true,"continua_vulneracion":true,"incumplimiento_normativo":true,"afectacion_individual":true,"medio_eficaz":false,"perjuicio_irremediable":true,"derecho":"salud","accionado":"EPS","pretension":"Programar la cirugía de vesícula"}
  estudiante: {"dano_consumado":true,"continua_vulneracion":true,"incumplimiento_normativo":true,"afectacion_individual":true,"otro_medio_judicial":true,"perjuicio_irremediable":true,"sujeto_especial_proteccion":true,"derecho":"salud","accionado":"EPS","pretension":"Ordenar la realización de la cirugía"} 

Le pedí a Colpensiones por escrito la historia laboral completa para revisar unas semanas que no me aparecen cotizadas. Van mes y medio y so
  profesor  : {"titular_pide_lo_suyo":true,"continua_vulneracion":true,"incumplimiento_normativo":true,"afectacion_individual":true,"otro_medi

## 4 · B3 · Medir con el harness de M2

`harness(eval_set, sistema)` de M2, para el profesor y para el estudiante, sobre **los mismos 130 casos** con los que se midió
el brazo 4 en M3 (`eval_set_v3` + `eval_set_m3`), **en la misma máquina**, con latencia por caso.
El profesor se mide con la caché vacía: con caché, la latencia mediría la caché.

Primero una prueba rápida con 10 casos; después los 130. A los dos sistemas se les vacía la caché antes de medir:
sin eso, los 10 casos de la prueba saldrían de la caché del estudiante en ~0 s. Si una corrida anterior quedó con ese
sesgo, `m5.medir(ctx, rehacer_estudiante=True)` mide de nuevo al estudiante (~3 min, sin Groq).

In [ ]:
m5.medir(ctx, n=10)
print(m5.tabla(ctx).T.to_string())

profesor: 0 ya medidos · faltan 10

estudiante: 0 ya medidos · faltan 10



modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

CONTRATO
  OK  acierto juridico       pedido: baja <= 5 puntos         medido: 60.0% -> 80.0% (-20.0 puntos perdidos)
  OK  alucinacion normativa  pedido: <= 0.0                   medido: 0.0
  OK  adversariales          pedido: baja <= 8 puntos         medido: 0.0% -> 0.0% (+0.0 puntos perdidos)
  NO  velocidad (mediana)    pedido: >= 3x mas rapido         medido: 6.65 s -> 2.75 s (2.4x)
  OK  estabilidad (p95)      pedido: p95 <= 5.0 s             medido: 3.05 s (profesor 9.27 s)
  tamaño: 237x menos parametros -> NO cumplido: lab: suban T (aqui: mas variedad de relatos), mas datos (N_POR_TEMA), o un estudiante mas grande (Qwen2.5-1.5B-Instruct)
modelo               profesor · openai/gpt-oss-120b (agente + reglas) estudiante · Qwen/Qwen2.5-0.5B-Instruct + LoRA (+ reglas)
acierto_juridico                                                  0.6                                                       0.8
acc_mecanismo                                                     0.6                   

In [ ]:
m5.medir(ctx)                              # los 130 (se retoma si se corta)

profesor: 0 ya medidos · faltan 130

estudiante: 0 ya medidos · faltan 130



{'profesor': {'salidas': ['Derecho: salud (art. 49 CP; art. 86 CP)\nAccionado: EPS\nMecanismo: acción de tutela\nProcedencia: sí — ninguna causal de improcedencia del artículo 6 resulta aplicable\nPretensión: Ordenar a la EPS suministrar el antirretroviral prescrito',
   'Derecho: salud (art. 49 CP; art. 86 CP; art. 8 Decreto 2591 de 1991)\nAccionado: hospital\nMecanismo: acción de tutela\nProcedencia: sí — aunque existe otro medio de defensa judicial, el perjuicio irremediable habilita la tutela como mecanismo transitorio\nPretensión: Garantizar la cirugía ortopédica necesaria',
   'Derecho: peticion (art. 23 CP; art. 6.1 Decreto 2591 de 1991)\nAccionado: empresa de acueducto\nMecanismo: vía judicial ordinaria\nProcedencia: no — existe otro medio de defensa judicial idóneo y eficaz en concreto, y no se acredita un perjuicio irremediable que habilite la tutela. No fue posible calcular el termino: no se encontró una fecha legible en el relato\nPretensión: Exigir explicación del aumento 

## 5 · La tabla antes/después y el contrato

In [ ]:
df = m5.tabla(ctx)
display(df.T)

CONTRATO
  NO  acierto juridico       pedido: baja <= 5 puntos         medido: 74.2% -> 45.0% (+29.2 puntos perdidos)
  OK  alucinacion normativa  pedido: <= 0.0                   medido: 0.0
  NO  adversariales          pedido: baja <= 8 puntos         medido: 88.1% -> 11.9% (+76.2 puntos perdidos)
  NO  velocidad (mediana)    pedido: >= 3x mas rapido         medido: 6.09 s -> 2.71 s (2.2x)
  OK  estabilidad (p95)      pedido: p95 <= 5.0 s             medido: 2.96 s (profesor 12.35 s)
  tamaño: 237x menos parametros -> NO cumplido: lab: suban T (aqui: mas variedad de relatos), mas datos (N_POR_TEMA), o un estudiante mas grande (Qwen2.5-1.5B-Instruct)


modelo,profesor · openai/gpt-oss-120b (agente + reglas),estudiante · Qwen/Qwen2.5-0.5B-Instruct + LoRA (+ reglas)
acierto_juridico,0.742,0.45
acc_mecanismo,0.742,0.45
acierto_gold,0.667,0.628
acierto_adversarial,0.881,0.119
acierto_abstencion,0.6,0.0
abstencion_indebida,0.008333,0.0
tasa_alucinacion,0.0,0.0
s_p50,6.09,2.71
s_p95,12.35,2.96
json_invalido,0,0


### Donde el estudiante falla y el profesor no (TODO 4)

Ahí está lo que la destilación no pudo copiar. Fíjense en tres cosas de la slide 19: ¿son adversariales (sesgos heredados)?,
¿categorías raras?, ¿se perdió la válvula de abstención (`acierto_abstencion` en la tabla)?

In [ ]:
perdidos = m5.casos_perdidos(ctx, mostrar=8)

60 casos donde el profesor acierta y el estudiante no (19 al reves). Por tipo: {'adversarial': 32, 'abstencion': 6, 'gold': 22}
  E10 [gold] gold via_ordinaria -> estudiante tutela: Compré una nevera en un almacén de cadena y me llegó golpeada. Ya van tres meses de reclamos, me dicen que el proveedor tiene que revisar el caso y nadie me dev
  A01 [adversarial] gold via_ordinaria -> estudiante tutela: La EPS me tuvo ocho meses esperando una cirugía de rodilla. Ya me la hicieron el mes pasado y salió bien, pero yo perdí el trabajo por las incapacidades y pasé 
  A03 [adversarial] gold via_ordinaria -> estudiante tutela: Fui a un cirujano plástico particular y me dio una orden para una abdominoplastia porque después de dos embarazos quedé con piel de más y eso me tiene con la au
  A04 [adversarial] gold via_ordinaria -> estudiante tutela: El juez laboral falló en contra mía y le creyó a los testigos de la empresa que claramente estaban mintiendo. Yo apelé y el tribunal confirmó. Esa sente

In [ ]:
m5.guardar(ctx)     # m5/ de esta copia del repo y entrega/ en Drive (incluye el manifiesto de la corrida)
m5.descargar(ctx)   # baja destilacion.json y pares_profesor.json: la copia de Colab se borra al cerrar

guardado:
  /content/Amparo/m5/destilacion.json
  /content/Amparo/m5/pares_profesor.json
  /content/drive/MyDrive/amparo_m5/entrega/destilacion.json
  /content/drive/MyDrive/amparo_m5/entrega/pares_profesor.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

['/content/drive/MyDrive/amparo_m5/entrega/destilacion.json',
 '/content/drive/MyDrive/amparo_m5/entrega/pares_profesor.json']

## Lo que va a M5

- **La tabla antes/después** de arriba (calidad con el harness de M2, latencia p50/p95, tamaño) y el veredicto del contrato.
- **El contrato** y **las licencias**: profesor gpt-oss-120b (Apache 2.0, permite entrenar con sus salidas) y estudiante Qwen2.5-0.5B (Apache 2.0).
- **Qué cedió el estudiante** (sección *limitaciones y ética*): los casos perdidos, la herramienta de términos (`calcular_termino`,
  que el estudiante no tiene) y la recuperación de normas (el estudiante no consulta el corpus: el fundamento lo pone el motor).
- En S15: el adaptador (`estudiante_lora/`) se cuantiza y se sube con el Space.

*Amparo · M5 · S14.*